Install required dependencies

In [17]:
%pip install langchain_community unstructured azure-identity python-dotenv langchain_huggingface langchain_openai azure-search-documents


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Import environment variables

In [15]:
from dotenv import load_dotenv
import os

load_dotenv()

VECTOR_SEARCH_ENDPOINT = os.getenv("VECTOR_SEARCH_ENDPOINT")
VECTOR_SEARCH_KEY = os.getenv("VECTOR_SEARCH_KEY")
AZURE_OPEN_API_ENDPOINT = os.getenv("AZURE_OPEN_API_ENDPOINT")
AZURE_OPEN_API_KEY = os.getenv("AZURE_OPEN_API_KEY")

Fetch the data

In [5]:
from langchain_community.document_loaders import UnstructuredURLLoader

urls = [
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ai-900',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ai-102',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ai-300',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/dp-100',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ab-731',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ab-100',
    'https://learn.microsoft.com/en-us/credentials/certifications/resources/study-guides/ab-730',
]

loader = UnstructuredURLLoader(urls)

documents = loader.load()

print(len(documents))

7


Split the data into chunks

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=20)

chunks = text_splitter.split_documents(documents)

print(len(chunks))

206


Store the data into vector database

In [18]:
from langchain_community.vectorstores.azuresearch import AzureSearch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import AzureOpenAIEmbeddings, OpenAIEmbeddings

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=AZURE_OPEN_API_ENDPOINT,
    api_key=AZURE_OPEN_API_KEY,
    azure_deployment="text-embedding-3-small",
)

vector_store = AzureSearch(
    azure_search_endpoint=VECTOR_SEARCH_ENDPOINT,
    azure_search_key=VECTOR_SEARCH_KEY,
    index_name='consine-similarity-demo',
    embedding_function=embeddings.embed_query
)

vector_store.add_documents(chunks)

['ZDA1NDJhZGMtYzZmZi00MTE3LWJkOWEtZGUzY2E0Nzk1YTQ3',
 'MWJmYzRlYmItMDhlYi00MzcyLWFlMzUtZTFmNGEzYjYwZDA3',
 'ZWQyNTllZTUtMTM1Zi00NTI5LWIwZjgtZGNjNjEwMzQyNzdk',
 'NmEwNTg0N2UtZTYwYy00ODE2LTk1Y2QtNWJmNWNjODNmOWQy',
 'NTFiNjA3MTItMjEyMi00NDU3LTlmNzQtYTVjMDQ3MTBjMGFj',
 'MmU4YTZmMWMtYzA0NC00YzlhLWFhZjMtZGNhNmU1NGIyMWVi',
 'ZTVhZjY5NDYtMjlkYy00ZmZiLTk1NzItMzAwNzVmNGYyNTc3',
 'ZWE3ZGZhYzAtNjZkNS00MzljLWIxNjgtNjllNGIwNWE3YWQ1',
 'M2IyYmRlZjgtOTg1Ni00ZjhjLTgxYjUtY2RlZDBkMmU2ZGQ2',
 'ZDRhNDkzNDAtM2Q3MC00OWY0LThiODctZTdlNDA1MjEwMzdj',
 'OWFkMzVlNDktZjIxYS00MGQ2LWEyZmItNWE1NDBkY2ZkMjEy',
 'NTJiNjA3ZDktODI0NC00ZjQ4LWJmNWUtYjkxYzA4ZjM4NzM5',
 'OTMxMTUzODMtYjBiZi00ZTJjLThjYzYtMjBjMTgyNDBhYjBm',
 'ZDU5ZTljODEtMWNlNC00ZjI3LWJiYmItMmZmYWRkYzY5MDNi',
 'NWJmZGM0YTAtYzc3Ny00ZmEwLWFhMzktOWFiNTM4NDVjZjIx',
 'NDJlM2IwNDctZGRlYS00ZWIyLTg4Y2MtODAxMWMxMjJlOWJk',
 'MDBiZDViZTEtYWI2OS00YzYzLTg5MzQtZWVjOTQ0NjVmZjk3',
 'ZGRmMGNiYWQtNGU4ZS00OGY1LTlmZjAtOGYxMTVlYTc3YTNm',
 'MWIwMWI1MmYtY2ExNi00MDk3LTkxNWYtYTU1ZTE3YjJj

User's query

In [19]:
user_query = "What resource should I refer to study for ai-900?"

Fetch the relevant chunk of data

In [20]:
retrieved_docs = vector_store.similarity_search(query=user_query, k=3)

retrieved_docs = " ".join([doc.page_content for doc in retrieved_docs])

Create prompt template

In [21]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["prompt", "relevant_chunk_of_data"],
    template=
    """
       You are an expert Microsoft Azure cloud instructor and mentor. Your task is to suggest practical project ideas that align with specific Microsoft Azure certification objectives. Base your recommendations strictly on the study guide content provided.

       Study Guide Content: {relevant_chunk_of_data}

       User Question: {prompt}

       Instructions for your response:
        1. Provide 3–5 project ideas relevant to the user’s question and the study guide content.
        2. For each project, include:
          - A clear project title
          - A brief description (2–3 sentences)
          - Which Azure services and tools would be involved
          - How it relates to the certification objectives
        3. Do not include information not covered in the study guide.
        4. Keep your language beginner-friendly and actionable.

       Format your response as a numbered list for clarity.
    """
)

Chain the prompt template with LLM and invoke it to generate the response

In [22]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPEN_API_ENDPOINT,
    api_key=AZURE_OPEN_API_KEY,
    azure_deployment="gpt-4o-mini",
    openai_api_version="2025-01-01-preview",
)

chain = prompt_template | llm

response = chain.invoke({
    "prompt": user_query,
    "relevant_chunk_of_data": retrieved_docs
})

print(response.content)

Here are some beginner-friendly project ideas that align with Microsoft Azure certification objectives, specifically for the AI-900 exam (Azure AI Fundamentals):

1. **Project Title: Simple Chatbot Deployment**
   - **Description**: Create a basic chatbot that can answer frequently asked questions for a fictional company. Utilize Azure's capabilities to design simple interactions and FAQs.
   - **Azure Services and Tools**: Use Azure Bot Service and QnA Maker to build and deploy the chatbot.
   - **Relation to Certification Objectives**: This project helps you understand how AI can enhance customer engagement and familiarity with deploying AI solutions using Azure tools, which are key objectives in the AI-900 certification.

2. **Project Title: Image Classifier with Azure Computer Vision**
   - **Description**: Develop a simple application that can analyze and categorize images based on predefined classes using Azure’s Computer Vision API. 
   - **Azure Services and Tools**: Leverage t